# Nettoyage table Sinistre

Bienvenue dans la partie Nettoyage de donnée.

Dans cette partie, je vais partir de la base de données fournie par mon professeur M.Manou-Abi. Cette base de données contient des informations sur les sinistres des clients d'une assurance factice nommée ENSAssuRances.

## Chargement des bibliothèques

Pour cela, je vais d'abord charger les bibliothèques dont j'aurai besoin :

In [1]:
import pandas as pd
import numpy as np

## Chargement des données

Je charge maintenant la table Sinistre pour le nettoyage :

In [2]:
sinistre = pd.read_csv('Sinistre.csv', sep=';', low_memory = False)

Voici un aperçu de la table :

In [3]:
sinistre.head(5)

,idx_sin,gar_sin,surv_sin,decl_sin,clo_sin,gest_sin,mt_eval,mt_regl
0,S18-0043146,AssVHR,2018-12-08,2018-12-10,NaN,2018-12-10,170.00,0.00
1,S18-0043146,AssVHR,2018-12-08,2018-12-10,NaN,2018-12-31,170.00,0.00
2,S18-0043146,AssVHR,2018-12-08,2018-12-10,2019-08-05,2019-08-05,262.86,262.86
3,S18-0057961,Ass0km,2018-10-30,2018-10-30,NaN,2018-10-30,150.00,0.00
4,S18-0057961,Ass0km,2018-10-30,2018-10-30,NaN,2018-12-31,150.00,0.00


## Nettoyage des données qualitatives

Les variables qualitatives de la table Sinistre doivent être transformées en texte et catégorie :

In [5]:
sinistre1 = sinistre.copy()

if 'idx_sin' in sinistre1.columns:
    sinistre1['idx_sin'] = sinistre1['idx_sin'].astype(str).replace('nan', np.nan)

if 'gar_sin' in sinistre1.columns:
    sinistre1['gar_sin'] = sinistre1['gar_sin'].astype('category')

In [ ]:
Je renomme ces variables pour plus de clarté :

In [6]:
sinistre2 = sinistre1.copy()

noms_colonnes_sin = {
    "idx_sin": "identifiant_sinistre",
    "gar_sin": "garantie_sinistre"
}

sinistre2 = sinistre2.rename(columns=noms_colonnes_sin)

print("Colonnes de sinistre2 après renommage :")
print(sinistre2.columns.tolist())

print("\n Aperçu des données :")
print(sinistre2[['identifiant_sinistre', 'garantie_sinistre']].head())

Colonnes de sinistre2 après renommage :
['identifiant_sinistre', 'garantie_sinistre', 'surv_sin', 'decl_sin', 'clo_sin', 'gest_sin', 'mt_eval', 'mt_regl']

 Aperçu des données :
  identifiant_sinistre garantie_sinistre
0          S18-0043146            AssVHR
1          S18-0043146            AssVHR
2          S18-0043146            AssVHR
3          S18-0057961            Ass0km
4          S18-0057961            Ass0km


Je souhaite voir les différentes valeurs que peut prendre la variable garantie_sinistre :

In [7]:
print("Liste des garanties présentes :")
print(sinistre2['garantie_sinistre'].unique())

Liste des garanties présentes :
['AssVHR', 'Ass0km', 'AssBase']
Categories (3, object): ['Ass0km', 'AssBase', 'AssVHR']


Je modifie ces valeurs pour plus de clarté :

In [8]:
sinistre3 = sinistre2.copy()

sinistre3['garantie_sinistre'] = sinistre3['garantie_sinistre'].astype(str)

mapping_garanties = {
    'Ass0km': 'Assistance 0 km',
    'AssBase': 'Assistance de base',
    'AssVHR': 'Assistance véhicule de remplacement'
}

sinistre3['garantie_sinistre'] = sinistre3['garantie_sinistre'].replace(mapping_garanties)

sinistre3['garantie_sinistre'] = sinistre3['garantie_sinistre'].astype('category')

print("Nouvelles modalités de la garantie :")
print(sinistre3['garantie_sinistre'].unique())

print("\nRépartition mise à jour :")
print(sinistre3['garantie_sinistre'].value_counts())

Nouvelles modalités de la garantie :
['Assistance véhicule de remplacement', 'Assistance 0 km', 'Assistance de base']
Categories (3, object): ['Assistance 0 km', 'Assistance de base', 'Assistance véhicule de remplacement']

Répartition mise à jour :
garantie_sinistre
Assistance de base                     57685
Assistance 0 km                         8575
Assistance véhicule de remplacement     5870
Name: count, dtype: int64


Les variables qualitatives sont maintenant propres.

## Nettoyage des données quantitatives

Les variables quantitatives doivent être en format numérique. Je renomme en même temps ces variables pour plus de clarté puis je demande un résumé :

In [9]:
sinistre4 = sinistre3.copy()

sinistre4['mt_eval'] = pd.to_numeric(sinistre4['mt_eval'], errors='coerce')
sinistre4['mt_regl'] = pd.to_numeric(sinistre4['mt_regl'], errors='coerce')

noms_montants = {
    "mt_eval": "montant_evaluation",
    "mt_regl": "montant_reglement"
}
sinistre4 = sinistre4.rename(columns=noms_montants)

print("Résumé des montants de sinistres :")
print(sinistre4[['montant_evaluation', 'montant_reglement']].describe())

Résumé des montants de sinistres :
       montant_evaluation  montant_reglement
count        72130.000000       72130.000000
mean           254.449224         157.651405
std            633.716936         661.917351
min           -369.990000        -369.990000
25%            170.000000           0.000000
50%            185.160000           0.000000
75%            215.000000         191.340000
max           7429.160000        7429.160000


Les variables quantitatives sont maintenant propres.

## Nettoyage des données temporelles

Les données temporelles doivent être en format date. Je renomme ces variables pour plus de clarté, je pose des conditions (un dossier ne peut être clôturé avant d'avoir été déclaré) puis je demande un résumé :

In [10]:
sinistre5 = sinistre4.copy()

noms_temporels = {
    "surv_sin": "date_survenance_sinistre",
    "decl_sin": "date_declaration_sinistre",
    "clo_sin": "date_cloture_sinistre"
}

sinistre5 = sinistre5.rename(columns=noms_temporels)

date_cols = ["date_survenance_sinistre", "date_declaration_sinistre", "date_cloture_sinistre"]

for col in date_cols:
    sinistre5[col] = pd.to_datetime(sinistre5[col], errors='coerce')

incoherences = sinistre5[sinistre5['date_cloture_sinistre'] < sinistre5['date_declaration_sinistre']]
print(f"Nombre de dossiers avec clôture avant déclaration : {len(incoherences)}")

print("\nRésumé des variables temporelles :")
print(sinistre5[date_cols].describe)

Nombre de dossiers avec clôture avant déclaration : 0

Résumé des variables temporelles :
<bound method NDFrame.describe of       date_survenance_sinistre date_declaration_sinistre date_cloture_sinistre
0                   2018-12-08                2018-12-10                   NaT
1                   2018-12-08                2018-12-10                   NaT
2                   2018-12-08                2018-12-10            2019-08-05
3                   2018-10-30                2018-10-30                   NaT
4                   2018-10-30                2018-10-30                   NaT
...                        ...                       ...                   ...
72125               2023-02-24                2023-02-25            2023-03-16
72126               2023-10-01                2023-10-02                   NaT
72127               2023-10-01                2023-10-02            2023-11-24
72128               2023-12-06                2023-12-07                   NaT
72129  

Les variables temporelles sont maintenant propres.

## Traitement des doublons

Je supprime maintenant les doublons stricts :

In [11]:
sinistre6 = sinistre5.copy()

nb_lignes_avant = len(sinistre6)

sinistre6 = sinistre6.drop_duplicates(keep='first')

nb_doublons_sin = nb_lignes_avant - len(sinistre6)

print(f"Nombre de doublons stricts supprimés dans la table Sinistre : {nb_doublons_sin}")
print(f"Nombre de lignes finales dans sinistre6 : {len(sinistre6)}")

Nombre de doublons stricts supprimés dans la table Sinistre : 68
Nombre de lignes finales dans sinistre6 : 72062


## Traitement des valeurs manquantes

Voyons maintenant le nombre de valeurs manquantes par colonnes :

In [12]:
missing_sin = sinistre6.isnull().sum()
missing_sin_pct = (sinistre6.isnull().sum() / len(sinistre6)) * 100

df_missing_sin = pd.DataFrame({
    'Valeurs Manquantes': missing_sin,
    'Pourcentage (%)': missing_sin_pct.round(2)
})

print("État des lieux des valeurs manquantes dans sinistre6 :")
print(df_missing_sin[df_missing_sin['Valeurs Manquantes'] > 0].sort_values(by='Valeurs Manquantes', ascending=False))

État des lieux des valeurs manquantes dans sinistre6 :
                       Valeurs Manquantes  Pourcentage (%)
date_cloture_sinistre               39031            54.16


Le fait que seule la variable date_cloture_sinistre ait des valeurs manquantes suggère qu'il s'agit de dossiers non clos. Ce sont probablement des sinistres qui viennent juste d'être déclarés (flux "frais"). L'expert n'a pas encore rendu son rapport (pas d'évaluation) et la compagnie n'a donc encore pas cloturé le dossier.

La table Sinistre est maintenant nettoyée.

In [13]:
sinistre6.to_csv('sinistre_cleaned.csv', index=False, sep=';', encoding='utf-8-sig')
print("Le fichier a été enregistré dans votre dossier de travail.")

Le fichier a été enregistré dans votre dossier de travail.
